# Bicing Barcelona — Anàlisi exploratòria de disponibilitat i patrons d'ús

**Projecte:** Itinerari d'Anàlisi de Dades · Cibernàrium / Barcelona Activa  
**Dades:** Open Data Barcelona · 291 milions de mesuraments · 548 estacions · 2020–2025

Aquest notebook realitza l'anàlisi exploratòria completa del servei Bicing:

1. **La xarxa** — mapa interactiu de les estacions i volum de viatges
2. **Patrons de demanda** — perfils horaris, setmanals i estacionals
3. **Disponibilitat** — quantificació del problema de les estacions buides i plenes

Cada secció genera els gràfics (PNG) i dades (CSV) necessaris per a l'article i la presentació del projecte. Tots els fitxers de sortida es guarden en subcarpetes: `png/`, `csv/`, `html/`.

---
## 0. Configuració

Importem les biblioteques, connectem a la base de dades DuckDB i definim els paràmetres globals que s'utilitzen al llarg de tot el notebook.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import folium
import json as _json
import os
import glob
import warnings
from datetime import date

warnings.filterwarnings('ignore')

# ── Estil global de Matplotlib ───────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'figure.dpi':       150,
    'savefig.dpi':      150,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

# Colors per als gràfics de disponibilitat
COLOR_BUIDA = '#d62728'   # vermell → estacions buides
COLOR_PLENA = '#1f77b4'   # blau    → estacions plenes

# ── Carpetes de sortida ──────────────────────────────────────────────────────
for carpeta in ['png', 'csv', 'html']:
    os.makedirs(carpeta, exist_ok=True)

# ── Connexió a DuckDB (només lectura) ───────────────────────────────────────
DB_PATH = 'bicicletes.duckdb'
con = duckdb.connect(DB_PATH, read_only=True)
print('Connexió establerta a', DB_PATH)

# ── Constants ────────────────────────────────────────────────────────────────
MESOS_CA = {
    1: 'Gen', 2: 'Feb', 3: 'Març', 4: 'Abr', 5: 'Maig', 6: 'Juny',
    7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Oct', 11: 'Nov', 12: 'Des'
}
MESOS_CA_LLARG = {
    1: 'Gener', 2: 'Febrer', 3: 'Març', 4: 'Abril', 5: 'Maig', 6: 'Juny',
    7: 'Juliol', 8: 'Agost', 9: 'Setembre', 10: 'Octubre', 11: 'Novembre', 12: 'Desembre'
}
DIES_CAT = {
    1: 'Dilluns', 2: 'Dimarts', 3: 'Dimecres', 4: 'Dijous',
    5: 'Divendres', 6: 'Dissabte', 7: 'Diumenge'
}

---
## 1. La xarxa Bicing — Mapa d'estacions

Visualització de les 548 estacions sobre el mapa de Barcelona. El radi de cada cercle és proporcional a la **mitjana de viatges mensuals** (2022–2025), de manera que les estacions amb més demanda destaquen visualment.

> **Diapositiva 2** de la presentació.

In [ ]:
# ── Dades per al mapa: mitjana mensual de viatges per estació ─────────────────
df_est = con.execute("""
    SELECT
        id_estacio,
        CAST(mitj_viatges_mes AS DOUBLE) AS mitj_viatges_mes,
        CAST(lat              AS DOUBLE) AS lat,
        CAST(lon              AS DOUBLE) AS lon,
        nom_districte
    FROM mitj_viatges_mes_estacio
    ORDER BY id_estacio
""").df()

# Normalitzar al rang [0, 1] per calcular el radi dels cercles
mv = df_est['mitj_viatges_mes']
mv_range = mv.max() - mv.min()
df_est['norm'] = (mv - mv.min()) / mv_range if mv_range > 0 else 0.5

# Radi inicial: escala sqrt amb factor 0.33 per evitar superposicions
MIN_R, MAX_R = 4.0, 20.0
df_est['radius'] = ((MIN_R + (MAX_R - MIN_R) * df_est['norm'] ** 0.5) * 0.33).round(2)

print(f'Estacions: {len(df_est)}')
print(f'Viatges/mes — mínim: {mv.min():.0f} · màxim: {mv.max():.0f} · mitjana: {mv.mean():.0f}')

In [ ]:
# ── Mapa interactiu amb Folium ────────────────────────────────────────────────
districtes = sorted(df_est['nom_districte'].unique())
paleta = [
    '#E63946', '#2A9D8F', '#E9C46A', '#264653', '#F4A261',
    '#457B9D', '#A8DADC', '#6A4C93', '#F77F00', '#023E8A',
    '#80B918', '#C77DFF'
]
mapa_colors = {d: paleta[i % len(paleta)] for i, d in enumerate(districtes)}
centre = [df_est['lat'].mean(), df_est['lon'].mean()]

m = folium.Map(location=centre, zoom_start=13,
               tiles='CartoDB positron',
               attr='© OpenStreetMap contributors, © CartoDB')

for _, fila in df_est.iterrows():
    color = mapa_colors[fila['nom_districte']]
    folium.CircleMarker(
        location=[fila['lat'], fila['lon']],
        radius=float(fila['radius']),
        color=color, fill=True, fill_color=color,
        fill_opacity=0.80, weight=1.0,
        tooltip=(
            f"<b>Estació {int(fila['id_estacio'])}</b><br>"
            f"Districte: {fila['nom_districte']}<br>"
            f"Viatges/mes: <b>{fila['mitj_viatges_mes']:.0f}</b>"
        )
    ).add_to(m)

# Títol i llegenda
m.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;top:16px;left:50%;transform:translateX(-50%);
    z-index:1000;background:white;padding:9px 22px;border-radius:9px;
    box-shadow:0 2px 8px rgba(0,0,0,.22);font-family:Arial,sans-serif;
    font-size:15px;font-weight:bold;color:#1D3557;pointer-events:none;">
  Bicing Barcelona — Estacions per districte (2022–2025)
</div>
"""))

elements_ll = ''.join(
    f'<li><span style="background:{mapa_colors[d]};width:12px;height:12px;'
    f'display:inline-block;border-radius:50%;margin-right:6px;"></span>{d}</li>'
    for d in districtes
)
m.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;bottom:40px;left:40px;z-index:1000;
    background:white;padding:14px 18px;border-radius:10px;
    box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:Arial,sans-serif;
    font-size:13px;line-height:1.7;">
  <b>Districtes</b>
  <ul style="list-style:none;margin:8px 0 0;padding:0;">{elements_ll}</ul>
  <hr style="margin:8px 0;">
  <small>{len(df_est)} estacions · radi ∝ √viatges/mes</small>
</div>
"""))

# Slider interactiu de radis
norms_list = df_est['norm'].round(5).tolist()
m.get_root().html.add_child(folium.Element(
    f'<script>window._bNorms = {_json.dumps(norms_list)};</script>'
))
m.get_root().html.add_child(folium.Element("""
<script>
window.addEventListener('load', function () {
  var MIN_R = 4.0, MAX_R = 20.0, norms = window._bNorms || [];
  var mapKey = Object.keys(window).find(function(k) {
    return k.indexOf('map_') === 0 && window[k] && window[k]._leaflet_id;
  });
  if (!mapKey) return;
  var lMap = window[mapKey], circles = [];
  lMap.eachLayer(function(l) { if (l instanceof L.CircleMarker) circles.push(l); });

  function update(f, e) {
    circles.forEach(function(c, i) {
      if (i < norms.length) c.setRadius((MIN_R + (MAX_R-MIN_R)*Math.pow(norms[i],e))*f);
    });
  }

  var p = L.control({position:'bottomright'});
  p.onAdd = function() {
    var d = L.DomUtil.create('div','');
    d.innerHTML =
      '<div style="background:white;padding:12px 16px;border-radius:10px;'+
      'box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:Arial;font-size:13px;min-width:200px;">'+
      '<b>Ajust dels cercles</b><br><br>'+
      'Factor: <b id="vF">0.33</b><br>'+
      '<input id="sF" type="range" min="0.05" max="2" step="0.05" value="0.33" style="width:100%;margin:4px 0 10px">'+
      '<br>Exponent: <b id="vE">0.50</b><br>'+
      '<input id="sE" type="range" min="0.1" max="2" step="0.05" value="0.5" style="width:100%;margin:4px 0">'+
      '</div>';
    d.querySelector('#sF').oninput = function(){
      document.getElementById('vF').textContent=parseFloat(this.value).toFixed(2);
      update(parseFloat(this.value),parseFloat(document.getElementById('sE').value));
    };
    d.querySelector('#sE').oninput = function(){
      document.getElementById('vE').textContent=parseFloat(this.value).toFixed(2);
      update(parseFloat(document.getElementById('sF').value),parseFloat(this.value));
    };
    L.DomEvent.disableClickPropagation(d);
    L.DomEvent.disableScrollPropagation(d);
    return d;
  };
  p.addTo(lMap);
});
</script>
"""))

m.save('html/bicicletes_mapa_estacions.html')
print('HTML guardat: html/bicicletes_mapa_estacions.html')
m

---
## 2. Evolució dels viatges mensuals (2020–2025)

Línia de temps amb el total de viatges per mes, des del gener de 2020 fins al darrer mes disponible. S'ombren els dos períodes de restriccions per COVID-19 que van afectar dràsticament la mobilitat.

> **Diapositiva 3** de la presentació.

In [ ]:
def formata_mes_ca(x, pos=None):
    """Eix X amb noms de mesos en català; any al gener."""
    dt = mdates.num2date(x)
    mes = MESOS_CA[dt.month]
    return f'{mes}\n{dt.year}' if dt.month == 1 else mes

# Consulta
df_timeline = con.execute("""
    SELECT any_, num_mes, SUM(CAST(viatges AS BIGINT)) AS total_viatges
    FROM viatges
    WHERE any_ >= 2020
    GROUP BY any_, num_mes
    ORDER BY any_, num_mes
""").df()

df_timeline['data'] = pd.to_datetime(
    df_timeline['any_'].astype(str) + '-'
    + df_timeline['num_mes'].astype(str).str.zfill(2) + '-01')
df_timeline = df_timeline.sort_values('data').reset_index(drop=True)

# Períodes COVID
covid = [
    {'i': date(2020,3,14), 'f': date(2020,6,21), 'c': '#E63946', 'a': .18,
     'l': '1r confinament estricte\n(14 mar – 21 jun 2020)', 'lx': date(2020,3,20), 'ly': .72},
    {'i': date(2020,10,25), 'f': date(2021,5,9), 'c': '#F4A261', 'a': .22,
     'l': "2n Estat d'Alarma\n(25 oct 2020 – 9 mai 2021)", 'lx': date(2020,11,1), 'ly': .54},
]

# Gràfic
fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#F8F9FA'); ax.set_facecolor('#F8F9FA')

ymax = df_timeline['total_viatges'].max()
for p in covid:
    ax.axvspan(pd.Timestamp(p['i']), pd.Timestamp(p['f']),
               color=p['c'], alpha=p['a'], linewidth=0)
    for edge in [p['i'], p['f']]:
        ax.axvline(pd.Timestamp(edge), color=p['c'], lw=1.2, ls='--', alpha=.7)
    ax.text(pd.Timestamp(p['lx']), ymax*p['ly'], p['l'],
            fontsize=8.5, color=p['c'], va='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=.3', fc='white', ec=p['c'], alpha=.85))

ax.fill_between(df_timeline['data'], df_timeline['total_viatges'],
                color='#457B9D', alpha=.15)
ax.plot(df_timeline['data'], df_timeline['total_viatges'],
        color='#1D3557', lw=2.2, marker='o', ms=4,
        mfc='white', mec='#1D3557', mew=1.4, label='Viatges / mes')

# Anotacions del mínim i màxim
for idx, color, etiq, yo in [
    (df_timeline['total_viatges'].idxmin(), '#E63946', 'Mínim', .25),
    (df_timeline['total_viatges'].idxmax(), '#2A9D8F', 'Màxim', .92)]:
    val = df_timeline.loc[idx, 'total_viatges']
    dt  = df_timeline.loc[idx, 'data']
    off = pd.DateOffset(months=-8) if etiq == 'Màxim' else pd.DateOffset(months=2)
    ax.annotate(f"{etiq}\n{val:,.0f}", xy=(dt, val),
                xytext=(dt + off, ymax * yo),
                fontsize=8.5, color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=color, lw=1.4),
                bbox=dict(boxstyle='round,pad=.3', fc='white', ec=color, alpha=.9))

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(formata_mes_ca))
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f} M' if x>=1e6 else f'{x/1e3:.0f} K'))
ax.set_ylim(0, ymax*1.12)
ax.yaxis.grid(True, ls='--', lw=.6, color='#CCC', alpha=.7)
ax.set_axisbelow(True)
ax.spines['left'].set_color('#CCC'); ax.spines['bottom'].set_color('#CCC')

ax.set_title('Bicicletes Barcelona — Viatges mensuals (gen 2020 – set 2025)',
             fontsize=14, fontweight='bold', pad=14, color='#1D3557')
ax.set_ylabel('Total de viatges', fontsize=10, color='#555')
patch1 = mpatches.Patch(color='#E63946', alpha=.45, label='1r confinament')
patch2 = mpatches.Patch(color='#F4A261', alpha=.55, label="2n Estat d'Alarma")
line_h = plt.Line2D([0],[0], color='#1D3557', lw=2, marker='o', ms=4, mfc='white',
                     label='Viatges mensuals')
ax.legend(handles=[line_h, patch1, patch2], loc='upper left', fontsize=9, framealpha=.9)

plt.tight_layout()
plt.savefig('png/bicicletes_viatges_mensuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/bicicletes_viatges_mensuals.png')

---
## 3. Patrons de demanda — Hora, dia i mes

### 3.1 Mapa de calor: viatges per hora i dia de la setmana

Matriu que mostra la mitjana de viatges a tota la xarxa per a cada combinació hora–dia. Permet identificar d'un cop d'ull els pics laborables (8 h i 18 h) i el patró diferent del cap de setmana.

> **Diapositiva 4** de la presentació (imatge esquerra).

In [ ]:
# Càrrega de la taula precalculada
df_heat = con.execute("""
    SELECT dia_setmana, hora_dia, mitj_viatges_hora_dia
    FROM mitj_viatges_hora_dia_setmana
    ORDER BY dia_setmana, hora_dia
""").df()

# Pivotar per al heatmap
pivot = df_heat.pivot(index='hora_dia', columns='dia_setmana', values='mitj_viatges_hora_dia')
pivot.columns = [DIES_CAT[c] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=.3, linecolor='white',
            annot=False, cbar_kws={'label': 'Viatges (mitjana)', 'shrink': .75})
ax.set_title('Viatges mitjans en tota la xarxa per hora i dia de la setmana · 2022–2025',
             fontsize=9, fontweight='bold', pad=10)
ax.set_xlabel('Dia de la setmana', fontsize=7)
ax.set_ylabel('Hora del dia', fontsize=7)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{int(h):02d}h' if int(h)%2==0 else '' for h in pivot.index],
                   rotation=0, fontsize=6)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=6.5)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=6); cbar.set_label('Viatges (mitjana)', fontsize=6)
fig.text(.5, -.03, 'Font: Open Data Barcelona · Servei Bicicletes · Elaboració pròpia',
         ha='center', fontsize=6, color='#666')

plt.tight_layout()
plt.savefig('png/bicicletes_heatmap_hora_dia.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/bicicletes_heatmap_hora_dia.png')

### 3.2 Estacionalitat mensual — Comparació entre anys

Total de viatges per mes, amb una línia per any. Permet veure el creixement interanual i la caiguda d'agost.

> **Diapositiva 4** (imatge dreta).

In [ ]:
df_mensual_p = con.execute("""
    SELECT any_, num_mes, viatges_mes
    FROM viatges_mes ORDER BY any_, num_mes
""").df()

ANY_CFG = {
    2022: {'color': '#9c27b0', 'ls': '-.',  'lw': 2.0, 'marker': 'D'},
    2023: {'color': '#1a6faf', 'ls': '-',   'lw': 2.3, 'marker': 'o'},
    2024: {'color': '#e07b39', 'ls': '--',  'lw': 2.3, 'marker': 's'},
    2025: {'color': '#4caf50', 'ls': ':',   'lw': 2.3, 'marker': '^'},
}

fig, ax = plt.subplots(figsize=(12, 6))
for any_num, cfg in ANY_CFG.items():
    sub = df_mensual_p[df_mensual_p['any_'] == any_num].sort_values('num_mes')
    if sub.empty: continue
    ax.plot(sub['num_mes'], sub['viatges_mes'], label=str(any_num),
            color=cfg['color'], ls=cfg['ls'], lw=cfg['lw'], marker=cfg['marker'], ms=7)
    idx_m = sub['viatges_mes'].idxmax()
    ax.annotate(f"{sub.loc[idx_m,'viatges_mes']/1e6:.1f} M",
                xy=(int(sub.loc[idx_m,'num_mes']), sub.loc[idx_m,'viatges_mes']),
                xytext=(0, 8), textcoords='offset points',
                fontsize=7.5, color=cfg['color'], ha='center')

ax.axvspan(7.5, 8.5, alpha=.07, color='#e07b39')
ax.text(8, ax.get_ylim()[0] or 0, 'Agost', ha='center', fontsize=7.5,
        color='#e07b39', va='bottom', style='italic')
ax.set_title('Estacionalitat mensual · Barcelona (2022–2025)',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Mes', fontsize=11)
ax.set_ylabel('Total de viatges', fontsize=11)
ax.set_xticks(range(1,13))
ax.set_xticklabels([MESOS_CA[m] for m in range(1,13)], fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f} M'))
ax.set_xlim(.5, 12.5)
ax.grid(axis='y', ls='--', alpha=.4)
ax.legend(title='Any', fontsize=10, loc='lower right', framealpha=.85)
fig.text(.5, -.04, 'Font: Open Data Barcelona · Servei Bicicletes · Elaboració pròpia',
         ha='center', fontsize=8, color='#666')

plt.tight_layout()
plt.savefig('png/bicicletes_mensual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/bicicletes_mensual.png')

### 3.3 Exportació CSV — dades de periodicitat

In [ ]:
# CSV 1: Perfil horari per dia de la setmana
df_csv_h = df_heat.copy()
df_csv_h['nom_dia_setmana'] = df_csv_h['dia_setmana'].map(DIES_CAT)
df_csv_h = df_csv_h[['dia_setmana', 'nom_dia_setmana', 'hora_dia', 'mitj_viatges_hora_dia']]
df_csv_h.to_csv('csv/bicicletes_viatges_hora_dia_setmana.csv', index=False, encoding='utf-8-sig')

# CSV 2: Total viatges per mes i any
df_csv_m = df_mensual_p.copy()
df_csv_m['nom_mes'] = df_csv_m['num_mes'].map(MESOS_CA_LLARG)
df_csv_m = df_csv_m[['any_', 'num_mes', 'nom_mes', 'viatges_mes']]
df_csv_m.to_csv('csv/bicicletes_viatges_per_mes.csv', index=False, encoding='utf-8-sig')

print(f'csv/bicicletes_viatges_hora_dia_setmana.csv  ({len(df_csv_h)} files)')
print(f'csv/bicicletes_viatges_per_mes.csv           ({len(df_csv_m)} files)')

---
## 4. Disponibilitat — El problema de les estacions buides i plenes

A partir d'aquí treballem amb la taula `totes_mitja_hores`, que conté una fila per cada combinació estació × franja de 30 minuts. Cada fila té un camp `estat` que classifica la franja com a `buida`, `plena`, `normal`, `extremes` o `sense_mesures`.

Les anàlisis d'aquesta secció exclouen el període 2020–2021 per evitar les distorsions de la pandèmia, llevat que s'indiqui el contrari.

### 4.1 Evolució anual


In [ ]:
df_anual = con.sql("""
    SELECT
        YEAR(data) AS any_,
        COUNT(*) FILTER (WHERE estat != 'sense_mesures')              AS slots_mesurats,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_plena,
        ROUND(COUNT(*) FILTER (WHERE estat IN ('buida','plena','extremes')) * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_problema,
        ROUND(COUNT(*) FILTER (WHERE estat = 'sense_mesures') * 100.0
            / COUNT(*), 2)                                            AS pct_sense_mesures
    FROM totes_mitja_hores
    GROUP BY YEAR(data)
    ORDER BY any_
""").df()

print(df_anual.to_string(index=False))
df_anual.to_csv('csv/01_evolucio_anual.csv', index=False)
print('\nCSV guardat: csv/01_evolucio_anual.csv')

### 4.2 Evolució mensual

Gràfic de línies que mostra l'evolució mes a mes del percentatge de franges buides i plenes. 

> **Diapositiva 5** de la presentació.

In [ ]:
df_mensual_d = con.sql("""
    SELECT
        YEAR(data) AS any_, MONTH(data) AS mes,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)   AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)   AS pct_plena,
        ROUND(COUNT(*) FILTER (WHERE estat IN ('buida','plena','extremes')) * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)   AS pct_problema
    FROM totes_mitja_hores
    GROUP BY YEAR(data), MONTH(data)
    ORDER BY any_, mes
""").df()
df_mensual_d['periode'] = df_mensual_d['any_'].astype(str) + '-' + df_mensual_d['mes'].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_mensual_d['periode'], df_mensual_d['pct_buida'],
        '-o', ms=3, label='Buida', color=COLOR_BUIDA)
ax.plot(df_mensual_d['periode'], df_mensual_d['pct_plena'],
        '-o', ms=3, label='Plena', color=COLOR_PLENA)
ax.plot(df_mensual_d['periode'], df_mensual_d['pct_problema'],
        '--', ms=2, label='Total problema', color='#333', alpha=.6)
ax.set_ylabel('% de franges de 30 min')
ax.set_title('Evolució mensual del percentatge de franges buides / plenes')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
step = max(1, len(df_mensual_d) // 20)
ax.set_xticks(range(0, len(df_mensual_d), step))
ax.set_xticklabels(df_mensual_d['periode'].iloc[::step], rotation=45, ha='right')
ax.grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig('png/02_evolucio_mensual.png')
plt.show()

df_mensual_d.to_csv('csv/02_evolucio_mensual.csv', index=False)
print('Guardat: png/02_evolucio_mensual.png + csv/02_evolucio_mensual.csv')

### 4.3 Patrons horaris

A quines hores del dia hi ha més estacions buides o plenes? Primer un gràfic global i després la mateixa informació desglossada per any.

> **Diapositiva 6** de la presentació.

In [ ]:
df_hora = con.sql("""
    SELECT hora_dia,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2) AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2) AS pct_plena
    FROM totes_mitja_hores
    GROUP BY hora_dia ORDER BY hora_dia
""").df()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_hora['hora_dia'] - .2, df_hora['pct_buida'], .4, label='Buida', color=COLOR_BUIDA)
ax.bar(df_hora['hora_dia'] + .2, df_hora['pct_plena'], .4, label='Plena', color=COLOR_PLENA)
ax.set_xlabel('Hora del dia'); ax.set_ylabel('% de franges de 30 min')
ax.set_title('Percentatge de franges buides / plenes per hora del dia (2022–2025)')
ax.set_xticks(range(24)); ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig('png/03_patrons_horaris.png')
plt.show()

df_hora.to_csv('csv/03_patrons_horaris.csv', index=False)
print('Guardat: png/03_patrons_horaris.png + csv/03_patrons_horaris.csv')

In [ ]:
# Mateixa informació, desglossada per any (dos panells)
df_hora_any = con.sql("""
    SELECT YEAR(data) AS any_, hora_dia,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2) AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2) AS pct_plena
    FROM totes_mitja_hores
    GROUP BY YEAR(data), hora_dia ORDER BY any_, hora_dia
""").df()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
anys = sorted(df_hora_any['any_'].unique())
colors_any = plt.cm.viridis([.1, .35, .6, .9])

for ax, col, titol in zip(axes, ['pct_buida','pct_plena'],
                           ['Franges BUIDES per hora', 'Franges PLENES per hora']):
    for j, a in enumerate(anys):
        sub = df_hora_any[df_hora_any['any_'] == a]
        ax.plot(sub['hora_dia'], sub[col], '-o', ms=3, label=str(a), color=colors_any[j])
    ax.set_xlabel('Hora del dia'); ax.set_ylabel('%'); ax.set_title(titol)
    ax.legend(fontsize=9); ax.set_xticks(range(0,24,2))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter()); ax.grid(alpha=.3)

plt.tight_layout()
plt.savefig('png/04_patrons_horaris_per_any.png')
plt.show()

df_hora_any.to_csv('csv/04_patrons_horaris_per_any.csv', index=False)
print('Guardat: png/04_patrons_horaris_per_any.png + csv/04_patrons_horaris_per_any.csv')

### 4.4 Mapa de calor: hora × dia de la setmana (disponibilitat)

Matriu 7×24 que mostra el percentatge de franges buides i plenes per cada combinació hora-dia.
Permet identificar els moments crítics de la setmana d'una ullada.

In [ ]:
df_heatmap = con.sql("""
    SELECT
        ISODOW(data) AS dia_setmana,
        hora_dia,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0 /
              NULLIF(COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 0), 2) AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0 /
              NULLIF(COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 0), 2) AS pct_plena
    FROM totes_mitja_hores
    GROUP BY ISODOW(data), hora_dia
    ORDER BY dia_setmana, hora_dia
""").df()
df_heatmap.to_csv('csv/06_heatmap_dia_hora.csv', index=False)

In [ ]:
import matplotlib.colors as mcolors

dies = ['Dl', 'Dt', 'Dc', 'Dj', 'Dv', 'Ds', 'Dg']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for metric, title, cmap, ax in zip(
    ['pct_buida', 'pct_plena'],
    ['% Franges BUIDES', '% Franges PLENES'],
    ['Reds', 'Blues'],
    axes):
    pivot = df_heatmap.pivot(index='dia_setmana', columns='hora_dia', values=metric)
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, interpolation='nearest')
    ax.set_yticks(range(7))
    ax.set_yticklabels(dies)
    ax.set_xticks(range(0, 24, 2))
    ax.set_xticklabels(range(0, 24, 2))
    ax.set_xlabel('Hora del dia')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='%', shrink=0.8)

plt.tight_layout()
plt.savefig('png/06_heatmap_dia_hora.png')
plt.show()

### 4.5 Districtes

Quins districtes de Barcelona pateixen més el problema?

> **Diapositiva 7** (imatge esquerra).

In [ ]:
df_dist = con.sql("""
    SELECT nom_districte,
        COUNT(*) FILTER (WHERE estat != 'sense_mesures')              AS slots_mesurats,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_plena,
        ROUND(COUNT(*) FILTER (WHERE estat IN ('buida','plena','extremes')) * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_problema
    FROM totes_mitja_hores
    GROUP BY nom_districte ORDER BY pct_problema DESC
""").df()

fig, ax = plt.subplots(figsize=(12, 6))
y = range(len(df_dist))
ax.barh(y, df_dist['pct_buida'], .4, label='Buida', color=COLOR_BUIDA, align='center')
ax.barh([i+.4 for i in y], df_dist['pct_plena'], .4, label='Plena', color=COLOR_PLENA, align='center')
ax.set_yticks([i+.2 for i in y]); ax.set_yticklabels(df_dist['nom_districte'])
ax.set_xlabel('% de franges de 30 min')
ax.set_title('Percentatge de franges buides / plenes per districte (2022–2025)')
ax.legend(); ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.invert_yaxis(); ax.grid(axis='x', alpha=.3)

plt.tight_layout()
plt.savefig('png/07_districtes.png')
plt.show()

df_dist.to_csv('csv/07_districtes.csv', index=False)
print('Guardat: png/07_districtes.png + csv/07_districtes.csv')

### 4.6 Distribució per estació — Histogrames

Com es distribueix el problema entre les 548 estacions? Tres histogrames mostren la dispersió del % de franges buides, plenes i amb problema total.

> **Diapositiva 7** (imatge dreta).

In [ ]:
df_estacio = con.sql("""
    SELECT id_estacio, nom_districte, lat, lon,
        COUNT(*) FILTER (WHERE estat != 'sense_mesures')              AS slots_mesurats,
        ROUND(AVG(avg_capacitat) FILTER (WHERE avg_capacitat IS NOT NULL), 1) AS capacitat_mitja,
        ROUND(COUNT(*) FILTER (WHERE estat = 'buida') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_buida,
        ROUND(COUNT(*) FILTER (WHERE estat = 'plena') * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_plena,
        ROUND(COUNT(*) FILTER (WHERE estat IN ('buida','plena','extremes')) * 100.0
            / COUNT(*) FILTER (WHERE estat != 'sense_mesures'), 2)    AS pct_problema
    FROM totes_mitja_hores
    GROUP BY id_estacio, nom_districte, lat, lon
    ORDER BY pct_problema DESC
""").df()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (col, xlabel, color) in zip(axes, [
    ('pct_buida', '% franges buides', COLOR_BUIDA),
    ('pct_plena', '% franges plenes', COLOR_PLENA),
    ('pct_problema', '% franges amb problema', '#7f7f7f')]):
    med = df_estacio[col].median()
    ax.hist(df_estacio[col], bins=40, color=color, alpha=.7, edgecolor='white')
    ax.axvline(med, color='black', ls='--', label=f'Mediana: {med:.1f}%')
    ax.set_xlabel(xlabel); ax.set_ylabel("Nombre d'estacions")
    ax.set_title(f'Distribució: {xlabel} per estació'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('png/11_distribucio_estacions.png')
plt.show()

df_estacio.to_csv('csv/11_estacions_problema.csv', index=False)
print(f'Guardat: png/11_distribucio_estacions.png + csv/11_estacions_problema.csv')
print(f'\nTop 10 estacions amb més problemes:')
print(df_estacio.head(10)[['id_estacio','nom_districte','pct_buida','pct_plena','pct_problema']].to_string(index=False))

### 4.7 Mapa interactiu — Problema dominant per estació

Cada cercle representa una estació. El **color** indica quin tipus de problema domina:
- 🟢 **Verd** → Poc problema (< 2%)
- 🔴 **Vermell** → Dominant BUIDA (l'estació es queda sense bicis)
- 🔵 **Blau** → Dominant PLENA (l'estació es queda sense anclatges)
- 🟠 **Taronja** → Mixt (ambdós problemes similars)

La **intensitat** del color reflecteix la severitat: com més fosc, pitjor.


In [ ]:
def color_estacio(pct_buida, pct_plena, pct_problema):
    """
    Retorna un color HTML segons el tipus i gravetat del problema.
    
    Lògica:
    - Si el problema total és petit (< 2%) → verd
    - Si domina 'buida' (més del doble que 'plena') → tons de vermell
    - Si domina 'plena' → tons de blau  
    - Si ambdós són similars → tons de taronja
    
    La intensitat (clar → fosc) depèn del % de problema total.
    """
    if pct_problema < 2:
        return '#66bb66'  # verd
    
    # Intensitat: 0 = suau, 1 = intens
    intensitat = min(pct_problema / 20, 1.0)
    
    if pct_buida > pct_plena * 2:
        # Dominant BUIDA → vermell (de rosa suau a vermell intens)
        r = int(230 + 25 * intensitat)  # 230 → 255
        g = int(160 - 160 * intensitat) # 160 → 0
        b = int(160 - 160 * intensitat) # 160 → 0
        return f'#{min(r,255):02x}{g:02x}{b:02x}'
    
    elif pct_plena > pct_buida * 2:
        # Dominant PLENA → blau (de blau suau a blau intens)
        r = int(160 - 140 * intensitat) # 160 → 20
        g = int(160 - 140 * intensitat) # 160 → 20
        b = int(200 + 55 * intensitat)  # 200 → 255
        return f'#{r:02x}{g:02x}{min(b,255):02x}'
    
    else:
        # MIXT → taronja (de groc suau a taronja intens)
        r = int(230 + 25 * intensitat)  # 230 → 255
        g = int(200 - 80 * intensitat)  # 200 → 120
        b = int(130 - 110 * intensitat) # 130 → 20
        return f'#{min(r,255):02x}{g:02x}{b:02x}'

print("Funció color_estacio() definida.")

In [ ]:
# Centre de Barcelona per al mapa
BCN_CENTER = [41.3925, 2.1654]

# Crear el mapa base centrat a Barcelona
mapa = folium.Map(location=BCN_CENTER, zoom_start=13, tiles='cartodbpositron')

# Afegir un cercle per cada estació (reutilitzem df_estacio calculat a 4.6)
for _, row in df_estacio.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue
    
    color = color_estacio(row['pct_buida'], row['pct_plena'], row['pct_problema'])
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=0.5,
        popup=folium.Popup(
            f"<b>Estació {int(row['id_estacio'])}</b><br>"
            f"{row['nom_districte']}<br>"
            f"Buida: {row['pct_buida']:.1f}% | Plena: {row['pct_plena']:.1f}%<br>"
            f"Total problema: {row['pct_problema']:.1f}%<br>"
            f"Capacitat: {row['capacitat_mitja']:.0f}",
            max_width=220
        ),
        tooltip=f"Est. {int(row['id_estacio'])}: {row['pct_problema']:.1f}% problema"
    ).add_to(mapa)

# Llegenda
llegenda = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 10px; border-radius: 5px;
     border: 2px solid grey; font-size: 12px; line-height: 1.8;">
<b>Problema dominant (2022–2025)</b><br>
<i style="background:#66bb66;width:12px;height:12px;display:inline-block;border-radius:50%"></i> Poc problema (&lt;2%)<br>
<i style="background:#cc1111;width:12px;height:12px;display:inline-block;border-radius:50%"></i> Dominant BUIDA<br>
<i style="background:#1144cc;width:12px;height:12px;display:inline-block;border-radius:50%"></i> Dominant PLENA<br>
<i style="background:#cc7711;width:12px;height:12px;display:inline-block;border-radius:50%"></i> Mixt
</div>
'''
mapa.get_root().html.add_child(folium.Element(llegenda))

# Guardar i mostrar
mapa.save('html/mapa_problema_combinat.html')
print("Mapa guardat: html/mapa_problema_combinat.html")
mapa

---
## 5. Tancament

Tanquem la connexió i llistem tots els fitxers generats.

In [ ]:
con.close()
print('Connexió a DuckDB tancada.\n')

print('=== Fitxers generats ===')
for carpeta in ['html', 'png', 'csv']:
    fitxers = sorted(glob.glob(f'{carpeta}/*'))
    if fitxers:
        print(f'\n{carpeta.upper()}/  ({len(fitxers)} fitxers)')
        for f in fitxers:
            print(f'  {f}  ({os.path.getsize(f)/1024:.0f} KB)')